# 🌵 Delhi Deep Dive — Module 01: Drought Risk
## Groundwater drought scoring for 11 Delhi districts (2010–2023)

In [ ]:
import os, time, warnings, pickle
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')

OUTPUT_DIR = '../data/outputs/delhi'
os.makedirs(OUTPUT_DIR, exist_ok=True)

START, END = '2010-01-01', '2023-12-31'
ARCHIVE_URL = 'https://archive-api.open-meteo.com/v1/archive'

DELHI_DISTRICTS = {
    'Central Delhi':    {'lat': 28.6508, 'lon': 77.2219},
    'East Delhi':       {'lat': 28.6271, 'lon': 77.2907},
    'New Delhi':        {'lat': 28.6139, 'lon': 77.2090},
    'North Delhi':      {'lat': 28.7041, 'lon': 77.1025},
    'North East Delhi': {'lat': 28.6921, 'lon': 77.2979},
    'North West Delhi': {'lat': 28.7219, 'lon': 77.0878},
    'Shahdara':         {'lat': 28.6735, 'lon': 77.2894},
    'South Delhi':      {'lat': 28.5244, 'lon': 77.1855},
    'South East Delhi': {'lat': 28.5677, 'lon': 77.2965},
    'South West Delhi': {'lat': 28.5672, 'lon': 77.0699},
    'West Delhi':       {'lat': 28.6541, 'lon': 77.0832},
}

def score_to_category(s):
    if s <= 25: return 'LOW'
    elif s <= 50: return 'MEDIUM'
    elif s <= 75: return 'HIGH'
    return 'VERY HIGH'

def fetch_with_retry(url, params, max_retries=5):
    for attempt in range(max_retries):
        r = requests.get(url, params=params, timeout=30)
        if r.status_code == 429:
            wait = 5 * (2 ** attempt)
            print(f'⏳ rate-limited, waiting {wait}s…', end=' ', flush=True)
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError('Max retries exceeded')

print('Setup complete. 11 districts, 2010–2023.')

## Cell 3 — Fetch Rainfall & ET₀ Data

In [ ]:
dfs = []
for district, coords in DELHI_DISTRICTS.items():
    print(f'  {district}…', end=' ', flush=True)
    params = {
        'latitude': coords['lat'], 'longitude': coords['lon'],
        'start_date': START, 'end_date': END,
        'daily': ['precipitation_sum', 'et0_fao_evapotranspiration'],
        'timezone': 'Asia/Kolkata'
    }
    data = fetch_with_retry(ARCHIVE_URL, params)['daily']
    df = pd.DataFrame({
        'date':       pd.to_datetime(data['time']),
        'rainfall_mm': pd.to_numeric(data['precipitation_sum'], errors='coerce').fillna(0),
        'et0_mm':     pd.to_numeric(data['et0_fao_evapotranspiration'], errors='coerce').fillna(0),
    })
    df['district'] = district
    df['lat'] = coords['lat']
    df['lon'] = coords['lon']
    dfs.append(df)
    print('✓')
    time.sleep(1.5)

daily = pd.concat(dfs, ignore_index=True)
daily['year']  = daily['date'].dt.year
daily['month'] = daily['date'].dt.month
print(f'\nTotal rows: {len(daily):,}  |  Districts: {daily["district"].nunique()}')
print(daily.head(3))

## Cell 5 — Yearly Feature Engineering

In [ ]:
def yearly_drought_features(g):
    g = g.sort_values('date')
    rain  = g['rainfall_mm']
    et0   = g['et0_mm']
    is_dry = (rain < 1.0)
    dry_spells = is_dry.groupby((~is_dry).cumsum()).sum()
    max_dry = int(dry_spells.max()) if len(dry_spells) > 0 else 0

    monthly = g.groupby('month')['rainfall_mm'].sum()
    monthly_full = pd.Series(0.0, index=range(1, 13))
    monthly_full.update(monthly)
    vals = monthly_full.values

    # Rolling 3-month SPI
    spi3_list = []
    for i in range(len(vals)):
        window = vals[max(0, i-2):i+1]
        if len(window) > 1 and np.std(window) > 0:
            spi3_list.append(stats.zscore(np.pad(window, (3-len(window), 0), mode='edge'))[-1])
        else:
            spi3_list.append(0.0)

    spi12 = float(stats.zscore(vals).mean()) if vals.std() > 0 else 0.0

    return pd.Series({
        'total_rainfall_mm':  float(rain.sum()),
        'total_et0_mm':       float(et0.sum()),
        'water_deficit_mm':   float(max(et0.sum() - rain.sum(), 0)),
        'dry_days':           int(is_dry.sum()),
        'max_dry_spell_days': max_dry,
        'spi_3month_mean':    float(np.mean(spi3_list)),
        'spi_12month_mean':   spi12,
    })

yearly = daily.groupby(['district', 'year']).apply(yearly_drought_features).reset_index()
for d, c in DELHI_DISTRICTS.items():
    yearly.loc[yearly['district'] == d, 'lat'] = c['lat']
    yearly.loc[yearly['district'] == d, 'lon'] = c['lon']

yearly['drought_label'] = (yearly['spi_12month_mean'] < -1.0).astype(int)
print(f'Yearly features shape: {yearly.shape}')
print(f'Drought events (label=1): {yearly["drought_label"].sum()} / {len(yearly)}')
print(yearly[['district','year','total_rainfall_mm','spi_12month_mean','drought_label']].head(11))

## Cell 7 — Train XGBoost Model

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

FEATURES = ['total_rainfall_mm', 'total_et0_mm', 'water_deficit_mm',
            'dry_days', 'max_dry_spell_days', 'spi_3month_mean']
TARGET   = 'drought_label'

train = yearly[yearly['year'] <= 2021].dropna(subset=FEATURES + [TARGET])
test  = yearly[yearly['year'] >= 2022].dropna(subset=FEATURES + [TARGET])

X_tr, y_tr = train[FEATURES].astype(float), train[TARGET].astype(int)
X_te, y_te = test[FEATURES].astype(float),  test[TARGET].astype(int)

model = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42)
model.fit(X_tr, y_tr)

y_pred = model.predict(X_te)
acc  = accuracy_score(y_te, y_pred)
prec = precision_score(y_te, y_pred, zero_division=0)
rec  = recall_score(y_te,  y_pred, zero_division=0)
f1   = f1_score(y_te,     y_pred, zero_division=0)
try:
    auc = roc_auc_score(y_te, model.predict_proba(X_te)[:,1])
except ValueError:
    auc = float('nan')
print(f'Accuracy: {acc:.3f}  AUC: {auc:.3f}  Precision: {prec:.3f}  Recall: {rec:.3f}  F1: {f1:.3f}')

# Feature importance
feat_df = pd.DataFrame({'feature': FEATURES, 'importance': model.feature_importances_}).sort_values('importance')
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(feat_df['feature'], feat_df['importance'], color='steelblue')
ax.set_title('Delhi Drought — Feature Importances')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/delhi_drought_feature_importance.png', dpi=150)
plt.show()

## Cell 9 — Score All Districts & Save

In [ ]:
full = yearly.dropna(subset=FEATURES + [TARGET]).copy()
probs = model.predict_proba(full[FEATURES].astype(float))[:, 1]
full['drought_prob']       = probs.round(4)
full['drought_risk_score'] = (probs * 100).round(2)
full['risk_category']      = full['drought_risk_score'].apply(score_to_category)

out_cols = ['district','year','lat','lon','drought_label','drought_prob','drought_risk_score','risk_category']
result = full[out_cols]
result.to_csv(f'{OUTPUT_DIR}/delhi_drought_scores.csv', index=False)
with open(f'{OUTPUT_DIR}/delhi_drought_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print(f'Saved delhi_drought_scores.csv ({len(result)} rows)')
print(result.groupby('risk_category')['district'].count())
print('\nSample output:')
print(result[result['year']==2023][['district','drought_risk_score','risk_category']].to_string())